# TMS Credential Management on DesignSafe

Before submitting jobs to TACC execution systems (Frontera, Stampede3, Lonestar6), you need **TMS credentials** established on each system. TMS (Trust Management System) manages SSH key pairs that allow Tapis to access these systems on your behalf.

This notebook shows how to:
1. Authenticate with DesignSafe
2. Check if TMS credentials exist on a system
3. Establish new credentials
4. Revoke credentials (for cleanup/reset)

## Install dapi

In [ ]:
%pip install dapi --quiet

## 1. Authenticate with DesignSafe

Initialize the `DSClient`. This handles authentication via environment variables, `.env` file, or interactive prompts.

In [ ]:
from dapi import DSClient, CredentialError

ds = DSClient()

## 2. Check existing credentials

Check whether TMS credentials already exist on a system. Returns `True` or `False`.

In [ ]:
# Check credentials on common TACC systems
systems = ["frontera", "stampede3", "ls6"]

for system_id in systems:
    has_creds = ds.systems.check_credentials(system_id)
    status = "ready" if has_creds else "needs setup"
    print(f"  {system_id}: {status}")

## 3. Establish TMS credentials

Establish credentials on systems where they are missing. This is **idempotent** -- if credentials already exist, it skips creation.

In [ ]:
# Establish credentials on all three systems
for system_id in systems:
    ds.systems.establish_credentials(system_id)

### Force re-creation

Use `force=True` to re-create credentials even if they already exist. This is useful if keys are corrupted or you need a fresh pair.

In [ ]:
# Force re-create credentials on a specific system
ds.systems.establish_credentials("frontera", force=True)

## 4. Verify setup and submit a job

After establishing credentials, verify they are in place and submit a test job.

In [ ]:
# Confirm all systems are ready
print("Credential status after setup:")
for system_id in systems:
    has_creds = ds.systems.check_credentials(system_id)
    print(f"  {system_id}: {'ready' if has_creds else 'MISSING'}")

## 5. Error handling

The `CredentialError` exception is raised when credential operations fail (e.g., system not found, non-TMS system).

In [ ]:
# Handling errors: non-existent system
try:
    ds.systems.establish_credentials("nonexistent-system")
except CredentialError as e:
    print(f"Expected error: {e}")

## 6. Revoke credentials (optional)

Remove TMS credentials from a system. This is useful for cleanup or resetting keys. Also idempotent -- succeeds silently if credentials don't exist.

In [ ]:
# Uncomment to revoke credentials on a system
# ds.systems.revoke_credentials("frontera")

## Summary

| Method | Purpose | Idempotent |
|--------|---------|------------|
| `ds.systems.check_credentials("system_id")` | Check if credentials exist | N/A (read-only) |
| `ds.systems.establish_credentials("system_id")` | Create credentials if missing | Yes |
| `ds.systems.establish_credentials("system_id", force=True)` | Re-create credentials | Yes |
| `ds.systems.revoke_credentials("system_id")` | Remove credentials | Yes |

All methods auto-detect your username from the authenticated Tapis client. You can also pass `username="other_user"` explicitly.